Task 1

In [1]:
import random

# Setting the seed for reproducibility
random.seed(42)

# Configuration for our synthetic data
categories = ["Technical", "Billing", "Hardware", "Account Access"]
raw_logs = []

for i in range(1, 201):
    # Determining the category (Technical is more common)
    cat = random.choices(categories, weights=[40, 20, 20, 20])[0]
    
    # Generating resolution minutes (with some "dirty" data)
    # 10% chance of being malformed/missing
    if random.random() < 0.10:
        res_min = random.choice([None, "NA", -5, "ERROR"])
    else:
        res_min = random.randint(5, 120)
    
    ticket = {
        "ticket_id": f"TKT-{1000 + i}",
        "customer_id": random.randint(500, 550), # 50 unique customers
        "category": cat,
        "resolution_minutes": res_min,
        "escalated": random.choice([True, False])
    }
    raw_logs.append(ticket)

# Validation: Printing count and first 5
print(f"Total tickets generated: {len(raw_logs)}")
print("Sample entries:")
for ticket in raw_logs[:5]:
    print(ticket)

Total tickets generated: 200
Sample entries:
{'ticket_id': 'TKT-1001', 'customer_id': 515, 'category': 'Hardware', 'resolution_minutes': -5, 'escalated': True}
{'ticket_id': 'TKT-1002', 'customer_id': 534, 'category': 'Technical', 'resolution_minutes': 99, 'escalated': True}
{'ticket_id': 'TKT-1003', 'customer_id': 513, 'category': 'Billing', 'resolution_minutes': None, 'escalated': True}
{'ticket_id': 'TKT-1004', 'customer_id': 545, 'category': 'Billing', 'resolution_minutes': 'NA', 'escalated': False}
{'ticket_id': 'TKT-1005', 'customer_id': 500, 'category': 'Technical', 'resolution_minutes': 108, 'escalated': True}


I used the random module with a fixed seed of 42 to ensure reproducibility. I generated 200 tickets as dictionaries within a list. To simulate "dirty" data, I introduced a 10% probability that the resolution_minutes field contains non-numeric or invalid values (None, string errors, or negatives). I also weighted the "Technical" category to appear more frequently to reflect a realistic support environment.

Task 2

In [2]:
def find_invalid_tickets(data_list):
    """Identifies tickets with missing or non-numeric resolution_minutes."""
    invalid_indices = []
    
    for index, record in enumerate(data_list):
        res = record.get("resolution_minutes")
        # Checking if it's not a number or if it's less than zero
        if not isinstance(res, (int, float)) or res < 0:
            invalid_indices.append(index)
            
    return invalid_indices

def check_missing_keys(data_list, required_keys):
    """Checks if any record is missing a mandatory key."""
    records_with_missing_keys = []
    
    for record in data_list:
        for key in required_keys:
            if key not in record:
                records_with_missing_keys.append(record["ticket_id"])
                break
                
    return records_with_missing_keys

# Testing the functions
bad_indices = find_invalid_tickets(raw_logs)
print(f"Found {len(bad_indices)} records with invalid resolution times.")

Found 19 records with invalid resolution times.


I created two validation helpers: find_invalid_tickets and check_missing_keys. These functions are designed to flag records that would cause errors in downstream analysis (like non-numeric values that break averages). By returning lists of indices or IDs instead of printing, these functions allow the pipeline to programmatically decide whether to repair or drop specific records.

Task 3

In [3]:
def clean_ticket_data(raw_data):
    cleaned_list = []
    
    for record in raw_data:
        # 1. Creating a shallow copy to avoid mutating the original
        clean_record = record.copy()
        
        # 2. Normalizing Category (Case and Whitespace)
        # e.g., " technical " -> "Technical"
        clean_record["category"] = str(clean_record["category"]).strip().title()
        
        # 3. Handling Resolution Minutes
        res = clean_record["resolution_minutes"]
        
        # If it's not a positive number, we'll "repair" it by setting it to 0 
        if not isinstance(res, (int, float)) or res < 0:
            clean_record["resolution_minutes"] = 0
        
        cleaned_list.append(clean_record)
        
    return cleaned_list

# Executing the cleaning
cleaned_logs = clean_ticket_data(raw_logs)

# Validation Check
print(f"Raw records: {len(raw_logs)}")
print(f"Cleaned records: {len(cleaned_logs)}")
print("\nFirst cleaned record:")
print(cleaned_logs[0])

Raw records: 200
Cleaned records: 200

First cleaned record:
{'ticket_id': 'TKT-1001', 'customer_id': 515, 'category': 'Hardware', 'resolution_minutes': 0, 'escalated': True}


My cleaning function performs two main operations:

Normalization: It ensures all categories are consistently formatted (e.g., "technical" becomes "Technical") using .strip().title().

Repairing Values: For records where resolution_minutes was missing or malformed (like "ERROR" or negative numbers), I decided to repair them by setting the value to 0. This allows us to keep the record for volume counting while acknowledging it has no valid resolution time. I used .copy() to ensure the raw_logs list remains unchanged for future comparison.

Task 4

In [4]:
# Importing the summary functions from our .py file
from m1_02_summary_functions import (
    get_average_res_time, 
    get_ticket_counts_by_customer, 
    get_escalation_metrics
)

# 1. Computing the summaries
avg_times = get_average_res_time(cleaned_logs)
cust_counts = get_ticket_counts_by_customer(cleaned_logs)
esc_report = get_escalation_metrics(cleaned_logs)

# 2. Displaying results for verification
print("--- Average Resolution Time per Category ---")
print(avg_times)
print("\n--- Escalation Metrics ---")
print(esc_report)

# 3. SANITY CHECKS
print("\n--- Running Sanity Checks ---")

# Check 1: Do the customer ticket totals match the cleaned record count?
total_tickets_check = sum(cust_counts.values()) == len(cleaned_logs)
print(f"Check 1 (Volume Match): {total_tickets_check}")

# Check 2: Are all categories accounted for in average times?
# (Assuming we have 4 categories)
cat_match_check = len(avg_times) == 4 
print(f"Check 2 (Category Count Match): {cat_match_check}")

# Check 3: Is the escalation rate a valid percentage (0.0 to 1.0)?
rate_check = 0 <= esc_report['overall_rate'] <= 1
print(f"Check 3 (Valid Escalation Rate): {rate_check}")

--- Average Resolution Time per Category ---
{'Hardware': 64.76086956521739, 'Technical': 63.46153846153846, 'Billing': 53.170731707317074, 'Account Access': 58.145833333333336}

--- Escalation Metrics ---
{'overall_rate': 0.51, 'by_category': {'Hardware': 0.5217391304347826, 'Technical': 0.5384615384615384, 'Billing': 0.5121951219512195, 'Account Access': 0.4583333333333333}}

--- Running Sanity Checks ---
Check 1 (Volume Match): True
Check 2 (Category Count Match): True
Check 3 (Valid Escalation Rate): True


In this section, I implemented three main summary functions:

Average Resolution Time: This uses a dictionary to accumulate both the total minutes and the count of tickets per category. This approach ensures that the denominator is correct for each category, even if some records were "repaired" (set to 0) during cleaning.

Customer Ticket Volume: This provides a count of interactions per customer ID, allowing us to identify high-frequency users.

Escalation Metrics: This calculates the percentage of tickets marked as 'Escalated' both globally and per category.

I have included Sanity Checks below each output to verify that the totals match my cleaned dataset of 200 records.

Task 5

In [5]:
def generate_final_report(avg_t, counts, esc):
    return {
        "summary_metrics": {
            "avg_resolution_by_category": avg_t,
            "escalation_data": esc
        },
        "volume_metrics": {
            "total_records": sum(counts.values()),
            "unique_customers": len(counts)
        }
    }

final_report = generate_final_report(avg_times, cust_counts, esc_report)

# Printing the report clearly
import json
print(json.dumps(final_report, indent=4))

{
    "summary_metrics": {
        "avg_resolution_by_category": {
            "Hardware": 64.76086956521739,
            "Technical": 63.46153846153846,
            "Billing": 53.170731707317074,
            "Account Access": 58.145833333333336
        },
        "escalation_data": {
            "overall_rate": 0.51,
            "by_category": {
                "Hardware": 0.5217391304347826,
                "Technical": 0.5384615384615384,
                "Billing": 0.5121951219512195,
                "Account Access": 0.4583333333333333
            }
        }
    },
    "volume_metrics": {
        "total_records": 200,
        "unique_customers": 51
    }
}


Insight: Based on the report, the "Technical" category not only has the highest volume of tickets but also shows a higher escalation rate compared to "Billing." This suggests we may need more senior staff or better documentation for technical issues.